In [ ]:
"""
Hybrid retrieval module.

Combines semantic search (vector DB) + BM25 (lexical) + cross-encoder reranking.
"""


def get_retriever():
    # TODO:
    # 1. Load vector store from disk (config.index_dir)
    # 2. Create semantic retriever from vector store
    # 3. Load chunks and create BM25 retriever
    # 4. Combine into ensemble retriever (semantic + BM25)
    # 5. Add cross-encoder reranker on top
    # 6. Return the final retriever
    pass

In [1]:
from llama_index.core import SimpleDirectoryReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os
import json

from config import Settings
settings = Settings()

def ingest():
   
    # 1. Load documents from config.data_dir (PDF, TXT, MD)
    documents = SimpleDirectoryReader(
    input_dir = settings.data_dir, 
    recursive=True,       #вкладені підпапки
    filename_as_id=True   #назву файлу як унікальний ідентифікатор (doc_id)
    ).load_data()

    # 2. Split into chunks using TextSplitter
    recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150, 
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

    langchain_docs = []
    for doc in documents:
        chunks = recursive_splitter.split_text(doc.text)
        for chunk in chunks:
            langchain_docs.append({
                "text": chunk,
                "metadata": doc.metadata    # Варіант із збереженням метаданих
            })
    
    texts = [doc["text"] for doc in langchain_docs]
    metadatas = [doc["metadata"] for doc in langchain_docs]

    # 3. Generate embeddings
    embeddings = OpenAIEmbeddings(
    api_key=settings.api_key.get_secret_value(),
    model=settings.embedding_model
    )

    # 4. Build vector store (FAISS, Qdrant, Chroma, etc.)
    if not os.path.exists(settings.index_dir):
        vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
   
        # 5. Save index to config.index_dir
        vectorstore.save_local(settings.index_dir)
    
        # 6. Save chunks for BM25 retriever (pickle or JSON)

        chunks_path = os.path.join(settings.index_dir, "chunks.json")
        with open(chunks_path, "w", encoding="utf-8") as f:
            json.dump(langchain_docs, f, ensure_ascii=False, indent=4)
        print(f"✅ Chunks saved: {chunks_path}")

c:\Users\User\Desktop\robotdream-course\agents\homework-lesson-3\.venv_hw3\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\User\Desktop\robotdream-course\agents\homework-lesson-3\.venv_hw3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

ingest()

2026-03-20 15:01:53,534 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-20 15:01:53,882 - INFO - Loading faiss with AVX2 support.
2026-03-20 15:01:53,906 - INFO - Successfully loaded faiss with AVX2 support.


✅ Chunks saved: index\chunks.json


In [4]:
# Load vector store from disk (config.index_dir)
from config import Settings
settings = Settings()

embeddings = OpenAIEmbeddings(
    api_key=settings.api_key.get_secret_value(),
    model=settings.embedding_model
    )

vectorstore = FAISS.load_local(settings.index_dir, embeddings, allow_dangerous_deserialization=True)

In [5]:
 # 2. Create semantic retriever from vector store
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": settings.retrieval_top_k})

In [6]:
# 3. Load chunks and create BM25 retriever
from langchain_community.retrievers import BM25Retriever

with open(os.path.join(settings.index_dir, "chunks.json"), "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

bm25_retriever = BM25Retriever.from_texts(
    texts=[doc["text"] for doc in raw_chunks],
    metadatas=[doc["metadata"] for doc in raw_chunks])
bm25_retriever.k = settings.retrieval_top_k

In [8]:
# 4. Combine into ensemble retriever (semantic + BM25)
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.6, 0.4] )
 

In [10]:
query = "ERROR-4532 authentication failure"
results = ensemble_retriever.invoke(query)
for i, doc in enumerate(results[:3]):
    print(f"Result {i+1}:")
    print(f"  Source: {doc.metadata.get('file_name', '?')}") 
    print(f"  Content: {doc.page_content[:150]}...")
    print()

2026-03-20 15:07:24,602 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Result 1:
  Source: large-language-model.pdf
  Content: risks across model architecture, training data, and deployment governance, and it emphasizes engineering
and policy interventions over media framings ...

Result 2:
  Source: large-language-model.pdf
  Content: A problem with the primitive dialog or task format is that users can create messages that appear to come
from the assistant or the developer. This may...

Result 3:
  Source: large-language-model.pdf
  Content: jailbreaking through carefully crafted user inputs that bypass safety training mechanisms.
Researchers from Anthropic found that it was possible to cr...



In [11]:
# 5. Add cross-encoder reranker on top

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor  = CrossEncoderReranker(
    model=reranker_model,
    top_n=settings.rerank_top_n
)


2026-03-20 15:16:30,832 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 15:16:30,881 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-03-20 15:16:30,933 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-03-20 15:16:31,245 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-03-20 15:16:31,447 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base/xet-read-token/2cfc18c9415c912f9d8155881c133215df768a70 "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5173.48it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: B

In [12]:
results = reranking_retriever.invoke("What loss function is used to train RAG?")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"  {doc.page_content[:150]}...")
    print()

2026-03-20 15:18:52,249 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:14<00:00, 14.48s/it]

Result 1:
  uploaded documents, or web sources.[1] According to Ars Technica, "RAG is a way of improving LLM
performance, in essence by blending the LLM process w...

Result 2:
  vector space. RAG can be used on unstructured (usually text),
semi-structured, or structured data (for example knowledge
graphs). These embeddings are...

Result 3:
  Finally, the LLM can generate output based on both the query and the retrieved documents.[2][6] Some
models incorporate extra steps to improve output,...



In [ ]:
# 6. Return the final retriever
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
)